In [ ]:
MICRODATI -DASH di esplorazione dati 4.0

In [ ]:
# variazione di 'Dash - esercizi - new - 5' in chiave 
# variazione di 'MICRODATI - esplorazione 2025 somministrazione - 1, 2,3,4,5'
# variazione di 'MICRODATI -DASH di esplorazione dati 2.0'
# variazione di 'MICRODATI -DASH di esplorazione dati 3.0'
# esplorazione somministrazione 2025 in MICRODATI
# 
#27/04/2026 versione funzionante ed abbastanza stabile
# BASE di sviluppo ulteriore

In [5]:
# ispirato da 
#https://github.com/plotly/tutorial-code/blob/main/session4/annex_b.py


#!pip install dash --upgrade
#!pip install dash-bootstrap-components

#1b. import libreries

#----------------------solo per WasmDash
#import micropip
#await micropip.install('dash-bootstrap-components')
#----------------------


import dash
import flask  # <--- Questo mancava!
import werkzeug

print(f"Dash version: {dash.__version__}")
print(f"Flask version: {flask.__version__}")
print(f"Werkzeug version: {werkzeug.__version__}")
#print(f"Dash_bootstrap_components version: {dash_bootstrap_components.__version__}")
#----------------

from dash import Dash, html, dcc, callback, Input, Output, State, no_update
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px

#------------------
%run ../../Libreria_JUPITER.ipynb
#run ../MICRODATI_funzioni.ipynb

Dash version: 4.1.0
Flask version: 2.3.2
Werkzeug version: 2.3.3


In [6]:
dir_save_dati= "C:\\Users\\lorid\\OneDrive\\Documenti\\Lavori_vari\\sistal_2\\eventi_SO\\eventi_SO_somministrazione_2025"
target_file=fr"{dir_save_dati}\\MICRO_SO_1525_5digit.csv"

df = pd.read_csv(target_file)

C:\Users\lorid\AppData\Local\Temp\ipykernel_18876\3588269314.py:4: DtypeWarning:

Columns (40,41,42) have mixed types. Specify dtype option on import or set low_memory=False.



In [7]:
import dash
from dash import dcc, html, Input, Output, callback, dash_table, State
import plotly.express as px
import pandas as pd
import dash_bootstrap_components as dbc
import os

# --- 0. PREPARAZIONE CARTELLA ESPORTAZIONE ---
export_dir = os.path.join("..", "esportazioni_quarto") 
if not os.path.exists(export_dir):
    os.makedirs(export_dir)

def force_utf8(filepath):
    try:
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='iso-8859-1') as f:
                content = f.read()
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)
    except:
        pass

# --- 1. CONFIGURAZIONE E DATI ---
MAPPA_NOMI_VISTE = {
    'nome_CPI_sede_lavoro': "Centro per l'Impiego",
    'qualifiche_cp2021_5digit': "Qualifica Professionale",
    'ateco2007_DIVISIONE_NOME': "Settore ATECO",
    'SESSO_LAV': "Genere",
    'ETA_LAV_ISTAT': "Fascia d'Età ISTAT",
    'TIPO_EVENTO': "Tipologia Evento",
    'NAZIONALITA': "Nazionalità",
    'ANNO_EVENTO': "Anno"
}

color_map_sesso = {'F': '#e74c3c', 'M': '#3498db'}

# Assumendo che df sia già caricato nel namespace
min_age, max_age = int(df['ETA_LAV'].min()), int(df['ETA_LAV'].max())

def get_options(col_name):
    options = [{'label': str(val), 'value': val} for val in sorted(df[col_name].unique())]
    if col_name == 'NAZIONALITA':
        special_options = [
            {'label': '-- SELEZIONA LE 10 PIÙ NUMEROSE --', 'value': 'TOP_10'},
            {'label': '-- SELEZIONA LE 25 PIÙ NUMEROSE --', 'value': 'TOP_25'}
        ]
        return special_options + options
    return options

def refine_fig_layout(fig, is_trend=False):
    titolo_legenda = fig.layout.legend.title.text if fig.layout.legend.title and fig.layout.legend.title.text else None
    if titolo_legenda:
        titolo_legenda = MAPPA_NOMI_VISTE.get(titolo_legenda, titolo_legenda.replace('_', ' '))

    for trace in fig.data:
        if trace.hovertemplate:
            for nome_grezzo, nome_bello in MAPPA_NOMI_VISTE.items():
                trace.hovertemplate = trace.hovertemplate.replace(nome_grezzo, nome_bello)
            trace.hovertemplate = trace.hovertemplate.replace('_', ' ')

    # Configurazione spaziale: l=80 per i trend per garantire allineamento verticale
    m_left, m_right, h_pref = (80, 50, 400) if is_trend else (220, 120, 750)

    fig.update_layout(
        xaxis_title=None, yaxis_title=None, height=h_pref,
        margin=dict(l=m_left, r=m_right, t=60, b=50),
        paper_bgcolor='white', plot_bgcolor='white',
        xaxis=dict(showgrid=True, gridcolor='rgba(235, 235, 235, 1)', zeroline=False),
        yaxis=dict(showgrid=True, gridcolor='rgba(235, 235, 235, 1)', zeroline=False),
        legend=dict(title_text=titolo_legenda, orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02, font=dict(size=10)),
        template="plotly_white"
    )
    return fig

# --- 2. LAYOUT ---
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dcc.Store(id='store-contratto-sel', data=None),
    dcc.Store(id='store-target-sel', data=None),
    html.H1('BI: Analisi Mercato del Lavoro', className="mt-3 text-primary fw-bold"),
    html.Hr(),
    dbc.Card([
        dbc.CardBody([
            dbc.Row([
                dbc.Col([html.Label("Anno"), dcc.Dropdown(id='f-anno', options=get_options('ANNO_EVENTO'), multi=True)], width=2),
                dbc.Col([html.Label("Sesso"), dcc.Dropdown(id='f-sesso', options=get_options('SESSO_LAV'), multi=True)], width=2),
                dbc.Col([html.Label("Tipo Evento"), dcc.Dropdown(id='f-evento', options=get_options('TIPO_EVENTO'), multi=True)], width=2),
                dbc.Col([html.Label("Nazionalità"), dcc.Dropdown(id='f-nazionalita', options=get_options('NAZIONALITA'), multi=True)], width=3),
                dbc.Col([html.Label("CPI"), dcc.Dropdown(id='f-cpi', options=get_options('nome_CPI_sede_lavoro'), multi=True)], width=3),
            ], className="mb-3"),
            dbc.Row([
                dbc.Col([
                    html.Label("Analisi Dettaglio:", className="fw-bold text-danger"),
                    dcc.RadioItems(id='target-dimension', options=[{'label': ' Qualifiche', 'value': 'qualifiche_cp2021_5digit'}, {'label': ' Settore ATECO', 'value': 'ateco2007_DIVISIONE_NOME'}], value='qualifiche_cp2021_5digit', inline=True)
                ], width=6),
                dbc.Col([html.Label("Età"), dcc.RangeSlider(id='f-eta', min=min_age, max=max_age, value=[min_age, max_age])], width=6),
            ])
        ])
    ], className="mb-4 shadow-sm border-0"),

    # Griglia Grafici
    dbc.Row([dbc.Col(id='card-1', children=dbc.Card([dbc.CardHeader("1. Contratti"), dbc.CardBody(dcc.Graph(id='graph-contracts'))]), width=6),
             dbc.Col(id='card-2', children=dbc.Card([dbc.CardHeader("2. Dettaglio"), dbc.CardBody(dcc.Graph(id='graph-target'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-3'), dbc.CardBody(dcc.Graph(id='trend-contracts-sesso'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-4'), dbc.CardBody(dcc.Graph(id='trend-target-sesso'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-5'), dbc.CardBody(dcc.Graph(id='trend-contracts-evento'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-6'), dbc.CardBody(dcc.Graph(id='trend-target-evento'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-7'), dbc.CardBody(dcc.Graph(id='trend-contracts-naz'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-8'), dbc.CardBody(dcc.Graph(id='trend-target-naz'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-9'), dbc.CardBody(dcc.Graph(id='trend-contracts-cpi'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-10'), dbc.CardBody(dcc.Graph(id='trend-target-cpi'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-11'), dbc.CardBody(dcc.Graph(id='trend-contracts-eta'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-12'), dbc.CardBody(dcc.Graph(id='trend-target-eta'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader("Riepilogo Numerico"), dbc.CardBody(id='table-container')]), width=12)])
], fluid=True)

# --- 3. LOGICA FILTRI ---
def get_filtered_df(anno, sesso, evento, naz, cpi, eta):
    dff = df.copy()
    if anno: dff = dff[dff['ANNO_EVENTO'].isin(anno)]
    if sesso: dff = dff[dff['SESSO_LAV'].isin(sesso)]
    if evento: dff = dff[dff['TIPO_EVENTO'].isin(evento)]
    if naz:
        if 'TOP_10' in naz or 'TOP_25' in naz:
            n = 10 if 'TOP_10' in naz else 25
            top_list = df['NAZIONALITA'].value_counts().nlargest(n).index.tolist()
            dff = dff[dff['NAZIONALITA'].isin(top_list)]
        else:
            dff = dff[dff['NAZIONALITA'].isin(naz)]
    if cpi: dff = dff[dff['nome_CPI_sede_lavoro'].isin(cpi)]
    if eta: dff = dff[(dff['ETA_LAV'] >= eta[0]) & (dff['ETA_LAV'] <= eta[1])]
    return dff

# --- 4. CALLBACKS ---

@callback(Output('store-contratto-sel', 'data'), Input('graph-contracts', 'clickData'), State('store-contratto-sel', 'data'))
def toggle_c(click, curr):
    if not click: return None
    new = click['points'][0]['y']
    return None if new == curr else new

@callback(Output('store-target-sel', 'data'), Input('graph-target', 'clickData'), State('store-target-sel', 'data'))
def toggle_t(click, curr):
    if not click: return None
    val = click['points'][0]['customdata'][0]
    return None if val == curr else val

@callback([Output('graph-contracts', 'figure'), Output('table-container', 'children')], 
          [Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value'), Input('store-contratto-sel', 'data')])
def update_g1(anno, sesso, evento, naz, cpi, eta, sel_c):
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    dff['VIZ'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
    counts = dff['VIZ'].value_counts()
    top = counts.nlargest(24).reset_index()
    top.columns = ['VIZ', 'count']
    if len(counts) > 24:
        top = pd.concat([top, pd.DataFrame({'VIZ': ['Altre Tipologie'], 'count': [counts.iloc[24:].sum()]})], ignore_index=True)
    top['percentage'] = (top['count'] / top['count'].sum() * 100).round(1)
    top['label_text'] = top.apply(lambda r: f"{r['count']} ({r['percentage']}%)", axis=1)
    
    fig = px.bar(top, y='VIZ', x='count', orientation='h', template="plotly_white", text='label_text', title="Distribuzione Contratti")
    fig.update_traces(marker_color=['#e74c3c' if v == sel_c else ('#2c3e50' if sel_c is None else '#bdc3c7') for v in top['VIZ']], textposition='auto')
    fig = refine_fig_layout(fig, is_trend=False)
    
    fig.write_html(f"{export_dir}/grafico_1.html", include_plotlyjs='cdn')
    fig.write_image(f"{export_dir}/grafico_1.png")
    force_utf8(f"{export_dir}/grafico_1.html")
    
    pivot = pd.crosstab(dff['VIZ'], dff['SESSO_LAV'], margins=True).reset_index()
    return fig, dash_table.DataTable(data=pivot.to_dict('records'), page_size=10)

@callback(
    Output('graph-target', 'figure'), 
    [Input('store-contratto-sel', 'data'), Input('target-dimension', 'value'), 
     Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), 
     Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value'), 
     Input('store-target-sel', 'data')]
)
def update_g2(sel_c, target_dim, anno, sesso, evento, naz, cpi, eta, sel_t):
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    
    # Filtro opzionale basato sulla selezione del Grafico 1
    if sel_c:
        dff['VIZ_C'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
        if sel_c == 'Altre Tipologie':
            top_24_esclusi = dff['VIZ_C'].value_counts().nlargest(24).index
            dff = dff[~dff['VIZ_C'].isin(top_24_esclusi)]
        else:
            dff = dff[dff['VIZ_C'] == sel_c]
    
    # Preparazione dati Top 25 (Qualifiche o ATECO)
    top = dff[target_dim].value_counts().nlargest(25).reset_index()
    top.columns = [target_dim, 'count']
    
    # Calcolo percentuali e testo etichette
    total_count = top['count'].sum()
    if total_count > 0:
        top['percentage'] = (top['count'] / total_count * 100).round(1)
        top['label_text'] = top.apply(lambda r: f"{r['count']} ({r['percentage']}%)", axis=1)
    else:
        top['label_text'] = ""

    # Tronca i nomi lunghi per la visualizzazione sull'asse Y
    top['VIZ_T'] = top[target_dim].apply(lambda x: (str(x)[:50] + '...') if len(str(x)) > 53 else x)
    
    # Creazione Grafico
    fig = px.bar(
        top, 
        y='VIZ_T', 
        x='count', 
        orientation='h', 
        template="plotly_white", 
        text='label_text', # <-- Visualizza etichette calcolate
        title=f"Dettaglio {MAPPA_NOMI_VISTE.get(target_dim)}", 
        custom_data=[target_dim]
    )
    
    # Logica colori e posizionamento testo
    colors = ['#e74c3c' if v == sel_t else ('#16a085' if sel_t is None else '#bdc3c7') for v in top[target_dim]]
    
    fig.update_traces(
        marker_color=colors, 
        textposition='auto', # <-- Posiziona il testo dentro o fuori le barre automaticamente
        hovertemplate="<b>%{customdata[0]}</b><br>Conteggio: %{x}<extra></extra>"
    )
    
    fig = refine_fig_layout(fig, is_trend=False)
    
    # Esportazione file
    try:
        fig.write_html(f"{export_dir}/grafico_2.html", include_plotlyjs='cdn')
        fig.write_image(f"{export_dir}/grafico_2.png")
        force_utf8(f"{export_dir}/grafico_2.html")
    except:
        pass
        
    return fig


@callback([Output('trend-contracts-sesso', 'figure'), Output('trend-contracts-evento', 'figure'), Output('trend-contracts-naz', 'figure'), Output('trend-contracts-cpi', 'figure'), Output('trend-contracts-eta', 'figure'), 
           Output('title-3', 'children'), Output('title-5', 'children'), Output('title-7', 'children'), Output('title-9', 'children'), Output('title-11', 'children')], 
          [Input('store-contratto-sel', 'data'), Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value')])
def update_t_left(sel_c, anno, sesso, evento, naz, cpi, eta):
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    
    ### NEW: Improved logic to prevent unnecessary title updates when toggling selection off
    if sel_c:
        dff['VIZ_C'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
        if sel_c == 'Altre Tipologie':
            top_24_esclusi = dff['VIZ_C'].value_counts().nlargest(24).index
            dff = dff[~dff['VIZ_C'].isin(top_24_esclusi)]
            label = "Altre Tipologie"
        else:
            dff = dff[dff['VIZ_C'] == sel_c]
            label = sel_c
    else:
        label = "Tutti i contratti"
    
    n_view = 25 if (naz and 'TOP_25' in naz) else 10
    
    f3 = px.line(dff.groupby(['ANNO_EVENTO', 'SESSO_LAV']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='SESSO_LAV', markers=True, color_discrete_map=color_map_sesso, title=f"Trend Genere: {label}")
    f5 = px.line(dff.groupby(['ANNO_EVENTO', 'TIPO_EVENTO']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='TIPO_EVENTO', markers=True, title=f"Trend Evento: {label}")
    f7 = px.line(dff[dff['NAZIONALITA'].isin(dff['NAZIONALITA'].value_counts().nlargest(n_view).index)].groupby(['ANNO_EVENTO', 'NAZIONALITA']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='NAZIONALITA', markers=True, title=f"Trend Top {n_view} Nazionalità: {label}")
    f9 = px.line(dff.groupby(['ANNO_EVENTO', 'nome_CPI_sede_lavoro']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='nome_CPI_sede_lavoro', markers=True, title=f"Trend CPI: {label}")
    f11 = px.line(dff.groupby(['ANNO_EVENTO', 'ETA_LAV_ISTAT']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='ETA_LAV_ISTAT', markers=True, title=f"Trend Età: {label}")
    
    for idx, f in enumerate([f3, f5, f7, f9, f11]):
        f = refine_fig_layout(f, is_trend=True)
        num = 3 + (idx * 2)
        try:
             f.write_html(f"{export_dir}/grafico_{num}.html", include_plotlyjs='cdn')
             f.write_image(f"{export_dir}/grafico_{num}.png")
             force_utf8(f"{export_dir}/grafico_{num}.html")
        except:
            pass
        
    return f3, f5, f7, f9, f11, f"3. {label}", f"5. {label}", f"7. {label}", f"9. {label}", f"11. {label}"


@callback([Output('trend-target-sesso', 'figure'), Output('trend-target-evento', 'figure'), Output('trend-target-naz', 'figure'), Output('trend-target-cpi', 'figure'), Output('trend-target-eta', 'figure'), 
           Output('title-4', 'children'), Output('title-6', 'children'), Output('title-8', 'children'), Output('title-10', 'children'), Output('title-12', 'children')], 
          [Input('store-target-sel', 'data'), Input('store-contratto-sel', 'data'), Input('target-dimension', 'value'), Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value')])
def update_t_right(sel_t, sel_c, target_dim, anno, sesso, evento, naz, cpi, eta):
    ### NEW: Input list expanded to include Input('store-contratto-sel', 'data')
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    
    ### NEW: Filter based on Graph 1 selection (Contract Type) FIRST
    base_label = ""
    if sel_c:
        dff['VIZ_C'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
        if sel_c == 'Altre Tipologie':
            top_24_esclusi = dff['VIZ_C'].value_counts().nlargest(24).index
            dff = dff[~dff['VIZ_C'].isin(top_24_esclusi)]
        else:
            dff = dff[dff['VIZ_C'] == sel_c]
        base_label = f"[{sel_c}] "
    
    ### NEW: Then, filter based on Graph 2 selection (Target Dimension)
    dim_bello = MAPPA_NOMI_VISTE.get(target_dim, target_dim)
    if sel_t: 
        dff = dff[dff[target_dim] == sel_t]
        label = f"{base_label}{sel_t}"
    else:
        top_25 = dff[target_dim].value_counts().nlargest(25).index
        dff = dff[dff[target_dim].isin(top_25)]
        label = f"{base_label}Top 25 {dim_bello}"
    
    n_view = 25 if (naz and 'TOP_25' in naz) else 10
    
    f4 = px.line(dff.groupby(['ANNO_EVENTO', 'SESSO_LAV']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='SESSO_LAV', markers=True, color_discrete_map=color_map_sesso, title=f"Trend Genere: {label}")
    f6 = px.line(dff.groupby(['ANNO_EVENTO', 'TIPO_EVENTO']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='TIPO_EVENTO', markers=True, title=f"Trend Evento: {label}")
    f8 = px.line(dff[dff['NAZIONALITA'].isin(dff['NAZIONALITA'].value_counts().nlargest(n_view).index)].groupby(['ANNO_EVENTO', 'NAZIONALITA']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='NAZIONALITA', markers=True, title=f"Trend Top {n_view} Nazionalità: {label}")
    f10 = px.line(dff.groupby(['ANNO_EVENTO', 'nome_CPI_sede_lavoro']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='nome_CPI_sede_lavoro', markers=True, title=f"Trend CPI: {label}")
    f12 = px.line(dff.groupby(['ANNO_EVENTO', 'ETA_LAV_ISTAT']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='ETA_LAV_ISTAT', markers=True, title=f"Trend Età: {label}")
    
    for idx, f in enumerate([f4, f6, f8, f10, f12]):
        f = refine_fig_layout(f, is_trend=True)
        num = 4 + (idx * 2)
        try:
             f.write_html(f"{export_dir}/grafico_{num}.html", include_plotlyjs='cdn')
             f.write_image(f"{export_dir}/grafico_{num}.png")
             force_utf8(f"{export_dir}/grafico_{num}.html")
        except:
             pass
        
    return f4, f6, f8, f10, f12, f"4. {label}", f"6. {label}", f"8. {label}", f"10. {label}", f"12. {label}"

if __name__ == '__main__':
    app.run(mode="inline", port=8058)

In [9]:
import dash
from dash import dcc, html, Input, Output, callback, dash_table, State
import plotly.express as px
import pandas as pd
import dash_bootstrap_components as dbc
import os

# --- 0. PREPARAZIONE CARTELLA ESPORTAZIONE ---
export_dir = os.path.join("..", "esportazioni_quarto") 
if not os.path.exists(export_dir):
    os.makedirs(export_dir)

def force_utf8(filepath):
    try:
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='iso-8859-1') as f:
                content = f.read()
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)
    except:
        pass

# --- 1. CONFIGURAZIONE E DATI ---
MAPPA_NOMI_VISTE = {
    'nome_CPI_sede_lavoro': "Centro per l'Impiego",
    'qualifiche_cp2021_5digit': "Qualifica Professionale",
    'ateco2007_DIVISIONE_NOME': "Settore ATECO",
    'SESSO_LAV': "Genere",
    'ETA_LAV_ISTAT': "Fascia d'Età ISTAT",
    'TIPO_EVENTO': "Tipologia Evento",
    'NAZIONALITA': "Nazionalità",
    'ANNO_EVENTO': "Anno"
}

color_map_sesso = {'F': '#e74c3c', 'M': '#3498db'}
color_map_evento = {
    'A': '#3498db', # Blu per Avviamenti
    'C': '#e74c3c', # Rosso per Cessazioni
    'P': '#00cc96', # Verde per Proroghe
    'T': '#ab63fa'  # Viola per Trasformazioni
}

# Assumendo che df sia già caricato nel namespace
min_age, max_age = int(df['ETA_LAV'].min()), int(df['ETA_LAV'].max())

def get_options(col_name):
    options = [{'label': str(val), 'value': val} for val in sorted(df[col_name].unique())]
    if col_name == 'NAZIONALITA':
        special_options = [
            {'label': '-- SELEZIONA LE 10 PIÙ NUMEROSE --', 'value': 'TOP_10'},
            {'label': '-- SELEZIONA LE 25 PIÙ NUMEROSE --', 'value': 'TOP_25'}
        ]
        return special_options + options
    return options

def refine_fig_layout(fig, is_trend=False):
    titolo_legenda = fig.layout.legend.title.text if fig.layout.legend.title and fig.layout.legend.title.text else None
    if titolo_legenda:
        titolo_legenda = MAPPA_NOMI_VISTE.get(titolo_legenda, titolo_legenda.replace('_', ' '))

    for trace in fig.data:
        if trace.hovertemplate:
            for nome_grezzo, nome_bello in MAPPA_NOMI_VISTE.items():
                trace.hovertemplate = trace.hovertemplate.replace(nome_grezzo, nome_bello)
            trace.hovertemplate = trace.hovertemplate.replace('_', ' ')

    m_left, m_right, h_pref = (80, 50, 400) if is_trend else (220, 120, 750)

    fig.update_layout(
        xaxis_title=None, yaxis_title=None, height=h_pref,
        margin=dict(l=m_left, r=m_right, t=60, b=50),
        paper_bgcolor='white', plot_bgcolor='white',
        xaxis=dict(showgrid=True, gridcolor='rgba(235, 235, 235, 1)', zeroline=False),
        yaxis=dict(showgrid=True, gridcolor='rgba(235, 235, 235, 1)', zeroline=False),
        legend=dict(title_text=titolo_legenda, orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02, font=dict(size=10)),
        template="plotly_white"
    )
    return fig

# --- 2. LAYOUT ---
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dcc.Store(id='store-contratto-sel', data=None),
    dcc.Store(id='store-target-sel', data=None),
    html.H1('BI: Analisi Mercato del Lavoro', className="mt-3 text-primary fw-bold"),
    html.Hr(),
    dbc.Card([
        dbc.CardBody([
            dbc.Row([
                dbc.Col([html.Label("Anno"), dcc.Dropdown(id='f-anno', options=get_options('ANNO_EVENTO'), multi=True)], width=2),
                dbc.Col([html.Label("Sesso"), dcc.Dropdown(id='f-sesso', options=get_options('SESSO_LAV'), multi=True)], width=2),
                dbc.Col([html.Label("Tipo Evento"), dcc.Dropdown(id='f-evento', options=get_options('TIPO_EVENTO'), multi=True)], width=2),
                dbc.Col([html.Label("Nazionalità"), dcc.Dropdown(id='f-nazionalita', options=get_options('NAZIONALITA'), multi=True)], width=3),
                dbc.Col([html.Label("CPI"), dcc.Dropdown(id='f-cpi', options=get_options('nome_CPI_sede_lavoro'), multi=True)], width=3),
            ], className="mb-3"),
            dbc.Row([
                dbc.Col([
                    html.Label("Analisi Dettaglio:", className="fw-bold text-danger"),
                    dcc.RadioItems(id='target-dimension', options=[{'label': ' Qualifiche', 'value': 'qualifiche_cp2021_5digit'}, {'label': ' Settore ATECO', 'value': 'ateco2007_DIVISIONE_NOME'}], value='qualifiche_cp2021_5digit', inline=True)
                ], width=6),
                dbc.Col([html.Label("Età"), dcc.RangeSlider(id='f-eta', min=min_age, max=max_age, value=[min_age, max_age])], width=6),
            ]),
            # --- NUOVO CONTROLLO: Switch per abilitare/disabilitare il filtro della colonna sinistra ---
            dbc.Row([
                dbc.Col([
                    html.Hr(),
                    dbc.Switch(
                        id="switch-filtro-sinistra",
                        label="Applica selezione del Grafico 1 anche ai grafici di sinistra (Trend Contratti)",
                        value=True, # Attivo di default
                        className="fw-bold text-info"
                    )
                ])
            ], className="mt-2")
        ])
    ], className="mb-4 shadow-sm border-0"),

    # Griglia Grafici
    dbc.Row([dbc.Col(id='card-1', children=dbc.Card([dbc.CardHeader("1. Contratti"), dbc.CardBody(dcc.Graph(id='graph-contracts'))]), width=6),
             dbc.Col(id='card-2', children=dbc.Card([dbc.CardHeader("2. Dettaglio"), dbc.CardBody(dcc.Graph(id='graph-target'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-3'), dbc.CardBody(dcc.Graph(id='trend-contracts-sesso'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-4'), dbc.CardBody(dcc.Graph(id='trend-target-sesso'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-5'), dbc.CardBody(dcc.Graph(id='trend-contracts-evento'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-6'), dbc.CardBody(dcc.Graph(id='trend-target-evento'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-7'), dbc.CardBody(dcc.Graph(id='trend-contracts-naz'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-8'), dbc.CardBody(dcc.Graph(id='trend-target-naz'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-9'), dbc.CardBody(dcc.Graph(id='trend-contracts-cpi'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-10'), dbc.CardBody(dcc.Graph(id='trend-target-cpi'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader(id='title-11'), dbc.CardBody(dcc.Graph(id='trend-contracts-eta'))]), width=6),
             dbc.Col(dbc.Card([dbc.CardHeader(id='title-12'), dbc.CardBody(dcc.Graph(id='trend-target-eta'))]), width=6)]),
    dbc.Row([dbc.Col(dbc.Card([dbc.CardHeader("Riepilogo Numerico"), dbc.CardBody(id='table-container')]), width=12)])
], fluid=True)

# --- 3. LOGICA FILTRI ---
def get_filtered_df(anno, sesso, evento, naz, cpi, eta):
    dff = df.copy()
    if anno: dff = dff[dff['ANNO_EVENTO'].isin(anno)]
    if sesso: dff = dff[dff['SESSO_LAV'].isin(sesso)]
    if evento: dff = dff[dff['TIPO_EVENTO'].isin(evento)]
    if naz:
        if 'TOP_10' in naz or 'TOP_25' in naz:
            n = 10 if 'TOP_10' in naz else 25
            top_list = df['NAZIONALITA'].value_counts().nlargest(n).index.tolist()
            dff = dff[dff['NAZIONALITA'].isin(top_list)]
        else:
            dff = dff[dff['NAZIONALITA'].isin(naz)]
    if cpi: dff = dff[dff['nome_CPI_sede_lavoro'].isin(cpi)]
    if eta: dff = dff[(dff['ETA_LAV'] >= eta[0]) & (dff['ETA_LAV'] <= eta[1])]
    return dff

# --- 4. CALLBACKS ---

@callback(Output('store-contratto-sel', 'data'), Input('graph-contracts', 'clickData'), State('store-contratto-sel', 'data'))
def toggle_c(click, curr):
    if not click: return None
    new = click['points'][0]['y']
    return None if new == curr else new

@callback(Output('store-target-sel', 'data'), Input('graph-target', 'clickData'), State('store-target-sel', 'data'))
def toggle_t(click, curr):
    if not click: return None
    val = click['points'][0]['customdata'][0]
    return None if val == curr else val

@callback([Output('graph-contracts', 'figure'), Output('table-container', 'children')], 
          [Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value'), Input('store-contratto-sel', 'data')])
def update_g1(anno, sesso, evento, naz, cpi, eta, sel_c):
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    dff['VIZ'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
    counts = dff['VIZ'].value_counts()
    top = counts.nlargest(24).reset_index()
    top.columns = ['VIZ', 'count']
    if len(counts) > 24:
        top = pd.concat([top, pd.DataFrame({'VIZ': ['Altre Tipologie'], 'count': [counts.iloc[24:].sum()]})], ignore_index=True)
    top['percentage'] = (top['count'] / top['count'].sum() * 100).round(1)
    top['label_text'] = top.apply(lambda r: f"{r['count']} ({r['percentage']}%)", axis=1)
    
    fig = px.bar(top, y='VIZ', x='count', orientation='h', template="plotly_white", text='label_text', title="Distribuzione Contratti")
    fig.update_traces(marker_color=['#e74c3c' if v == sel_c else ('#2c3e50' if sel_c is None else '#bdc3c7') for v in top['VIZ']], textposition='auto')
    fig = refine_fig_layout(fig, is_trend=False)
    
    fig.write_html(f"{export_dir}/grafico_1.html", include_plotlyjs='cdn')
    fig.write_image(f"{export_dir}/grafico_1.png")
    force_utf8(f"{export_dir}/grafico_1.html")
    
    pivot = pd.crosstab(dff['VIZ'], dff['SESSO_LAV'], margins=True).reset_index()
    return fig, dash_table.DataTable(data=pivot.to_dict('records'), page_size=10)

@callback(
    Output('graph-target', 'figure'), 
    [Input('store-contratto-sel', 'data'), Input('target-dimension', 'value'), 
     Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), 
     Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value'), 
     Input('store-target-sel', 'data')]
)
def update_g2(sel_c, target_dim, anno, sesso, evento, naz, cpi, eta, sel_t):
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    
    if sel_c:
        dff['VIZ_C'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
        if sel_c == 'Altre Tipologie':
            top_24_esclusi = dff['VIZ_C'].value_counts().nlargest(24).index
            dff = dff[~dff['VIZ_C'].isin(top_24_esclusi)]
        else:
            dff = dff[dff['VIZ_C'] == sel_c]
    
    top = dff[target_dim].value_counts().nlargest(25).reset_index()
    top.columns = [target_dim, 'count']
    
    total_count = top['count'].sum()
    if total_count > 0:
        top['percentage'] = (top['count'] / total_count * 100).round(1)
        top['label_text'] = top.apply(lambda r: f"{r['count']} ({r['percentage']}%)", axis=1)
    else:
        top['label_text'] = ""

    top['VIZ_T'] = top[target_dim].apply(lambda x: (str(x)[:50] + '...') if len(str(x)) > 53 else x)
    
    fig = px.bar(
        top, y='VIZ_T', x='count', orientation='h', template="plotly_white", 
        text='label_text', title=f"Dettaglio {MAPPA_NOMI_VISTE.get(target_dim)}", custom_data=[target_dim]
    )
    
    colors = ['#e74c3c' if v == sel_t else ('#16a085' if sel_t is None else '#bdc3c7') for v in top[target_dim]]
    
    fig.update_traces(
        marker_color=colors, textposition='auto', 
        hovertemplate="<b>%{customdata[0]}</b><br>Conteggio: %{x}<extra></extra>"
    )
    
    fig = refine_fig_layout(fig, is_trend=False)
    
    try:
        fig.write_html(f"{export_dir}/grafico_2.html", include_plotlyjs='cdn')
        fig.write_image(f"{export_dir}/grafico_2.png")
        force_utf8(f"{export_dir}/grafico_2.html")
    except:
        pass
        
    return fig


@callback([Output('trend-contracts-sesso', 'figure'), Output('trend-contracts-evento', 'figure'), Output('trend-contracts-naz', 'figure'), Output('trend-contracts-cpi', 'figure'), Output('trend-contracts-eta', 'figure'), 
           Output('title-3', 'children'), Output('title-5', 'children'), Output('title-7', 'children'), Output('title-9', 'children'), Output('title-11', 'children')], 
          [Input('store-contratto-sel', 'data'), Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value'), 
           Input('switch-filtro-sinistra', 'value')]) # <-- Input per lo switch aggiunto
def update_t_left(sel_c, anno, sesso, evento, naz, cpi, eta, applica_filtro):
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    
    # --- LOGICA DI OVERRIDE SWITCH ---
    # Se lo switch è disattivato, forziamo il comportamento come se nessuna barra fosse selezionata
    active_sel_c = sel_c if applica_filtro else None
    
    if active_sel_c:
        dff['VIZ_C'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
        if active_sel_c == 'Altre Tipologie':
            top_24_esclusi = dff['VIZ_C'].value_counts().nlargest(24).index
            dff = dff[~dff['VIZ_C'].isin(top_24_esclusi)]
            label = "Altre Tipologie"
        else:
            dff = dff[dff['VIZ_C'] == active_sel_c]
            label = active_sel_c
    else:
        label = "Tutti i contratti"
    
    n_view = 25 if (naz and 'TOP_25' in naz) else 10
    
    f3 = px.line(dff.groupby(['ANNO_EVENTO', 'SESSO_LAV']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='SESSO_LAV', markers=True, color_discrete_map=color_map_sesso, title=f"Trend Genere: {label}")
    f5 = px.line(dff.groupby(['ANNO_EVENTO', 'TIPO_EVENTO']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='TIPO_EVENTO', markers=True, color_discrete_map=color_map_evento, title=f"Trend Evento: {label}")
    f7 = px.line(dff[dff['NAZIONALITA'].isin(dff['NAZIONALITA'].value_counts().nlargest(n_view).index)].groupby(['ANNO_EVENTO', 'NAZIONALITA']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='NAZIONALITA', markers=True, title=f"Trend Top {n_view} Nazionalità: {label}")
    f9 = px.line(dff.groupby(['ANNO_EVENTO', 'nome_CPI_sede_lavoro']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='nome_CPI_sede_lavoro', markers=True, title=f"Trend CPI: {label}")
    f11 = px.line(dff.groupby(['ANNO_EVENTO', 'ETA_LAV_ISTAT']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='ETA_LAV_ISTAT', markers=True, title=f"Trend Età: {label}")
    
    for idx, f in enumerate([f3, f5, f7, f9, f11]):
        f = refine_fig_layout(f, is_trend=True)
        num = 3 + (idx * 2)
        try:
             f.write_html(f"{export_dir}/grafico_{num}.html", include_plotlyjs='cdn')
             f.write_image(f"{export_dir}/grafico_{num}.png")
             force_utf8(f"{export_dir}/grafico_{num}.html")
        except:
            pass
        
    return f3, f5, f7, f9, f11, f"3. {label}", f"5. {label}", f"7. {label}", f"9. {label}", f"11. {label}"


@callback([Output('trend-target-sesso', 'figure'), Output('trend-target-evento', 'figure'), Output('trend-target-naz', 'figure'), Output('trend-target-cpi', 'figure'), Output('trend-target-eta', 'figure'), 
           Output('title-4', 'children'), Output('title-6', 'children'), Output('title-8', 'children'), Output('title-10', 'children'), Output('title-12', 'children')], 
          [Input('store-target-sel', 'data'), Input('store-contratto-sel', 'data'), Input('target-dimension', 'value'), Input('f-anno', 'value'), Input('f-sesso', 'value'), Input('f-evento', 'value'), Input('f-nazionalita', 'value'), Input('f-cpi', 'value'), Input('f-eta', 'value')])
def update_t_right(sel_t, sel_c, target_dim, anno, sesso, evento, naz, cpi, eta):
    dff = get_filtered_df(anno, sesso, evento, naz, cpi, eta)
    
    base_label = ""
    if sel_c:
        dff['VIZ_C'] = dff['TIPOLOGIA_CONTRATTUALE'].apply(lambda x: (str(x)[:40] + '...') if len(str(x)) > 43 else x)
        if sel_c == 'Altre Tipologie':
            top_24_esclusi = dff['VIZ_C'].value_counts().nlargest(24).index
            dff = dff[~dff['VIZ_C'].isin(top_24_esclusi)]
        else:
            dff = dff[dff['VIZ_C'] == sel_c]
        base_label = f"[{sel_c}] "
    
    dim_bello = MAPPA_NOMI_VISTE.get(target_dim, target_dim)
    if sel_t: 
        dff = dff[dff[target_dim] == sel_t]
        label = f"{base_label}{sel_t}"
    else:
        top_25 = dff[target_dim].value_counts().nlargest(25).index
        dff = dff[dff[target_dim].isin(top_25)]
        label = f"{base_label}Top 25 {dim_bello}"
    
    n_view = 25 if (naz and 'TOP_25' in naz) else 10
    
    f4 = px.line(dff.groupby(['ANNO_EVENTO', 'SESSO_LAV']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='SESSO_LAV', markers=True, color_discrete_map=color_map_sesso, title=f"Trend Genere: {label}")
    f6 = px.line(dff.groupby(['ANNO_EVENTO', 'TIPO_EVENTO']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='TIPO_EVENTO', markers=True, title=f"Trend Evento: {label}")
    f8 = px.line(dff[dff['NAZIONALITA'].isin(dff['NAZIONALITA'].value_counts().nlargest(n_view).index)].groupby(['ANNO_EVENTO', 'NAZIONALITA']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='NAZIONALITA', markers=True, title=f"Trend Top {n_view} Nazionalità: {label}")
    f10 = px.line(dff.groupby(['ANNO_EVENTO', 'nome_CPI_sede_lavoro']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='nome_CPI_sede_lavoro', markers=True, title=f"Trend CPI: {label}")
    f12 = px.line(dff.groupby(['ANNO_EVENTO', 'ETA_LAV_ISTAT']).size().reset_index(name='C'), x='ANNO_EVENTO', y='C', color='ETA_LAV_ISTAT', markers=True, title=f"Trend Età: {label}")
    
    for idx, f in enumerate([f4, f6, f8, f10, f12]):
        f = refine_fig_layout(f, is_trend=True)
        num = 4 + (idx * 2)
        try:
             f.write_html(f"{export_dir}/grafico_{num}.html", include_plotlyjs='cdn')
             f.write_image(f"{export_dir}/grafico_{num}.png")
             force_utf8(f"{export_dir}/grafico_{num}.html")
        except:
             pass
        
    return f4, f6, f8, f10, f12, f"4. {label}", f"6. {label}", f"8. {label}", f"10. {label}", f"12. {label}"

if __name__ == '__main__':
    app.run(mode="inline", port=8060)